# Simple EDA — operating-room data

This notebook explores `donees bloc anonyme pour centrale 2026.xlsx`. It covers data quality, activity over time, patient age, length of stay, intervention/anesthesia categories, and operating-room timing.

> Privacy: patient, case, and practitioner identifiers are excluded from row-level previews and charts.

In [ ]:
# If needed, run once:
# %pip install pandas openpyxl matplotlib seaborn

from pathlib import Path
from datetime import datetime, time
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
DATA_FILE = Path("donees bloc anonyme pour centrale 2026.xlsx")
assert DATA_FILE.exists(), f"File not found: {DATA_FILE.resolve()}"

## 1. Load and prepare the data

In [ ]:
df_raw = pd.read_excel(DATA_FILE, engine="openpyxl")
print(f"Rows: {len(df_raw):,} | Columns: {df_raw.shape[1]}")
display(pd.DataFrame({"column": df_raw.columns, "dtype": df_raw.dtypes.astype(str).values}))

In [ ]:
def clean_name(name):
    text = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

df = df_raw.copy()
df.columns = [clean_name(c) for c in df.columns]

date_cols = ["date_entree", "date_sortie", "date_naissance", "date_inter"]
for col in date_cols:
    if col in df:
        df[col] = pd.to_datetime(df[col], errors="coerce")

safe_preview = df.drop(columns=["no_cas", "id_patient", "praticien", "nom_chir"], errors="ignore")
display(safe_preview.head())

## 2. Data quality

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100).round(1),
    "unique_n": df.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)

print(f"Exact duplicate rows: {df.duplicated().sum():,}")
display(quality)

In [ ]:
missing = quality.query("missing_pct > 0").sort_values("missing_pct")
if not missing.empty:
    ax = missing["missing_pct"].plot.barh(figsize=(9, max(4, len(missing) * 0.28)), color="#4C78A8")
    ax.set(title="Missing values by column", xlabel="Missing (%)", ylabel="")
    plt.tight_layout()
    plt.show()

## 3. Derived measures

In [ ]:
if {"date_naissance", "date_inter"}.issubset(df.columns):
    df["age_years"] = (df["date_inter"] - df["date_naissance"]).dt.days / 365.25
    df.loc[~df["age_years"].between(0, 110), "age_years"] = np.nan

stay_col = next((c for c in df.columns if c.startswith("duree_sejour")), None)
if stay_col:
    df["length_of_stay_days"] = pd.to_numeric(df[stay_col], errors="coerce")
    df.loc[df["length_of_stay_days"] < 0, "length_of_stay_days"] = np.nan

def clock_to_minutes(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (datetime, time)):
        return value.hour * 60 + value.minute + value.second / 60
    if isinstance(value, (int, float, np.number)):
        return float(value) * 24 * 60
    parsed = pd.to_datetime(str(value), errors="coerce")
    return np.nan if pd.isna(parsed) else parsed.hour * 60 + parsed.minute + parsed.second / 60

time_candidates = {
    "sspi_pre": next((c for c in df if c.startswith("heure_entree_sspi")), None),
    "room_in": next((c for c in df if c.startswith("heure_d_entree_en_salle")), None),
    "incision": next((c for c in df if c.startswith("heure_incision")), None),
    "room_out": next((c for c in df if c.startswith("heure_de_sortie_de_salle")), None),
}
for short_name, col in time_candidates.items():
    if col:
        df[f"{short_name}_minute"] = df[col].map(clock_to_minutes)

def elapsed_minutes(end, start):
    delta = end - start
    return delta.where(delta >= 0, delta + 24 * 60)

if {"room_in_minute", "room_out_minute"}.issubset(df.columns):
    df["room_duration_min"] = elapsed_minutes(df["room_out_minute"], df["room_in_minute"])
if {"room_in_minute", "incision_minute"}.issubset(df.columns):
    df["entry_to_incision_min"] = elapsed_minutes(df["incision_minute"], df["room_in_minute"])
if {"sspi_pre_minute", "room_in_minute"}.issubset(df.columns):
    df["preop_wait_min"] = elapsed_minutes(df["room_in_minute"], df["sspi_pre_minute"])

for col in ["room_duration_min", "entry_to_incision_min", "preop_wait_min"]:
    if col in df:
        df.loc[~df[col].between(0, 24 * 60), col] = np.nan

## 4. Numerical overview

In [ ]:
measure_cols = [c for c in [
    "age_years", "length_of_stay_days", "room_duration_min",
    "entry_to_incision_min", "preop_wait_min"
] if c in df]
display(df[measure_cols].describe(percentiles=[.25, .5, .75, .9, .95]).T.round(1))

In [ ]:
if measure_cols:
    fig, axes = plt.subplots(len(measure_cols), 1, figsize=(9, 3.2 * len(measure_cols)))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, measure_cols):
        upper = df[col].quantile(0.99)
        sns.histplot(df.loc[df[col].between(0, upper), col], bins=35, ax=ax, color="#59A14F")
        ax.set_title(f"Distribution of {col} (up to 99th percentile)")
    plt.tight_layout()
    plt.show()

## 5. Activity over time

In [ ]:
if "date_inter" in df:
    monthly = df.dropna(subset=["date_inter"]).set_index("date_inter").resample("MS").size()
    ax = monthly.plot(figsize=(11, 4), marker="o", markersize=3, color="#F28E2B")
    ax.set(title="Interventions per month", xlabel="Month", ylabel="Number of interventions")
    plt.tight_layout()
    plt.show()

    weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    weekday = df["date_inter"].dt.day_name().value_counts().reindex(weekday_order)
    ax = weekday.plot.bar(figsize=(9, 4), color="#E15759")
    ax.set(title="Interventions by weekday", xlabel="", ylabel="Number of interventions")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 6. Main categorical variables

In [ ]:
categorical_cols = [c for c in [
    "interv_type", "anesth_type", "anesth_loco_reg",
    "sexe", "ghm_code", "cim_diag_pr", "ccam_1"
] if c in df]

for col in categorical_cols:
    counts = df[col].fillna("Missing").astype(str).value_counts().head(15).sort_values()
    ax = counts.plot.barh(figsize=(9, max(3, len(counts) * 0.32)), color="#76B7B2")
    ax.set(title=f"Top categories: {col}", xlabel="Number of records", ylabel="")
    plt.tight_layout()
    plt.show()

## 7. Duration by intervention type

In [ ]:
if {"interv_type", "room_duration_min"}.issubset(df.columns):
    common_types = df["interv_type"].value_counts().head(12).index
    plot_data = df[df["interv_type"].isin(common_types)].copy()
    plot_data = plot_data[plot_data["room_duration_min"] <= plot_data["room_duration_min"].quantile(.99)]
    order = plot_data.groupby("interv_type")["room_duration_min"].median().sort_values().index
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=plot_data, y="interv_type", x="room_duration_min", order=order, showfliers=False)
    plt.title("Operating-room duration for the 12 most frequent intervention types")
    plt.xlabel("Room duration (minutes)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

## 8. Compact summary table

Use this table to identify common procedures with long room occupancy and enough observations for reliable comparison.

In [ ]:
if {"interv_type", "room_duration_min"}.issubset(df.columns):
    summary = (
        df.groupby("interv_type", dropna=False)
          .agg(
              interventions=("interv_type", "size"),
              median_room_min=("room_duration_min", "median"),
              p90_room_min=("room_duration_min", lambda s: s.quantile(.9)),
              median_entry_to_incision_min=("entry_to_incision_min", "median"),
          )
          .sort_values("interventions", ascending=False)
    )
    display(summary.head(20).round(1))

## Notes for interpretation

- Missing timestamps and zero clock values should be checked before operational conclusions are drawn.
- The timing calculations assume a procedure that crosses midnight lasts less than 24 hours.
- Associations in this exploratory notebook are descriptive, not causal.
- Small intervention groups should not be compared without reporting their sample sizes.